In [387]:
from google.cloud import bigquery
from google.cloud.exceptions import NotFound
import pickle

# Loading Sample Data and CPC/IPC Codes

In [20]:
data = pickle.load(open('data/multilingual_sample_1000en500nen_perCat_10encoders.pckl','rb'))

In [21]:
client  = bigquery.Client()

In [60]:
patent_ids = f"'{data[0]['patent_id']}'"
for item in data:
    patent_ids += f""",'{item['patent_id']}'"""

query_text = f"""SELECT patent_id, cpcs, ipcs FROM unified_patents.classifications WHERE patent_id IN ({patent_ids})"""

In [88]:
query_job = client.query(query_text)
query_results = query_job.result()
cpc_ipc_codes = [row for row in query_results]

In [91]:
codes = [{'patent_id':item[0], 'cpcs':item[1], 'ipcs':item[2]} for item in cpc_ipc_codes]

In [101]:
for item in data:
    item['cpcs'] = []
    item['ipcs'] = []

In [102]:
for item in data:
    for test in codes:
        if item['patent_id'] == test['patent_id']:
            item['cpcs'] = test['cpcs']
            item['ipcs'] = test['ipcs']

# Finding Patent Families

In [118]:
families = {}
for item in data:
    family_id = item['family_id']
    try:
        families[family_id].append(item)
    except KeyError:
        families[family_id] = [item]

In [452]:
family_ids = [key for key in families.keys()]

In [131]:
cpc_codes = []
ipc_codes = []
for family_id in family_ids:
    for item in families[family_id]:
        cpc_codes.extend(item['cpcs'])
        ipc_codes.extend(item['ipcs'])

In [185]:
cpc_code_list = f"'{cpc_codes[0]}'"
cpc_code_high_level = f"'{cpc_codes[0].split()[0]}'"
for code in cpc_codes:
    code_high_level = code.split()[0]
    code_reformat = code.replace(" ","").replace("/","-")
    cpc_code_list += f", '{code_reformat}'"
    cpc_code_high_level += f", '{code_high_level}'"

In [432]:
query_text = f"SELECT code, description FROM kq57_sandbox.code_descriptions_3 WHERE code IN ({cpc_code_list})"
query_job = client.query(query_text)
query_results = query_job.result()

In [433]:
cpc_code_titles = [row for row in query_results]
cpc_titles = [{'cpc':item[0], 'title':item[1]} for item in cpc_code_titles]

In [442]:
query_text = f"SELECT code, description FROM kq57_sandbox.code_descriptions_3 WHERE code IN ({cpc_code_high_level})"
query_job = client.query(query_text)
query_results = query_job.result()

In [438]:
cpc_code_high_level_titles = [row for row in query_results]
#cpc_high_level_titles = [{'cpc':item[0], 'title':item[1]} for item in cpc_code_high_level_titles]

In [439]:
cpc_high_level_titles = [{'cpc':item[0], 'title':item[1]} for item in cpc_code_high_level_titles]

In [440]:
cpc_high_level_titles

[{'cpc': 'A01B',
  'title': 'SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS, DETAILS, OR ACCESSORIES OF AGRICULTURAL MACHINES OR IMPLEMENTS, IN GENERAL  (making or covering furrows or holes for sowing, planting, or manuring A01C5/00; soil working for engineering purposes E01, E02, E21; {measuring areas for agricultural purposes G01B})'},
 {'cpc': 'A01C',
  'title': 'PLANTING; SOWING; FERTILISING  (parts, details or accessories of agricultural machines or implements, in general A01B51/00\xa0-\xa0A01B75/00)'},
 {'cpc': 'A01D', 'title': 'HARVESTING; MOWING'},
 {'cpc': 'A01G',
  'title': 'HORTICULTURE; CULTIVATION OF VEGETABLES, FLOWERS, RICE, FRUIT, VINES, HOPS OR SEAWEED; FORESTRY; WATERING  (picking of fruits, vegetables, hops or the like A01D46/00; propagating unicellular algae C12N1/12)'},
 {'cpc': 'A01H',
  'title': 'NEW PLANTS OR {NON-TRANSGENIC} PROCESSES FOR OBTAINING THEM; PLANT REPRODUCTION BY TISSUE CULTURE TECHNIQUES'},
 {'cpc': 'A01J',
  'title': 'MANUFACTURE OF DAIRY PRODUCT

In [457]:
families[family_ids[0]][0].keys()

dict_keys(['patent_id', 'family_id', 'category', 'title_abstract', 'title_abstract_original', 'language', 'encoding1', 'encoding2', 'encoding3', 'encoding4', 'encoding5', 'cpcs', 'ipcs'])

In [458]:
family_list = []
for family_id in family_ids:
    patents = families[family_id]
    lang = ''
    title_abstract_to_embed = ''

In [459]:
patents

[{'patent_id': 'CN-111598740-A',
  'family_id': '72188398',
  'category': 'real_estate',
  'title_abstract': 'Method for tracking latest position of property equipment maintenance personnel The invention belongs to the technical field of online management of property equipment maintenance personnel, and discloses a latest position tracking method for the property equipment maintenance personnel. The method comprises the steps that a user obtains GSP data and personnel information of the maintenance personnel, and stores the data; current user permission configuration information is inquired in a database, permission control is conducted on a current user, related operation buttons are displayed for the user with permission to check personnel position information, and personnel position information safety is ensured; querying the latest positions of all personnel in the maintenance project at the current time according to the currently selected maintenance project, and displaying whethe